# UniverSat Models Usage

This notebook demonstrates how to use the UniverSat model (Resolution- and Modality-Agnostic Transformers for Earth Observation) via the `geoai` wrapper. The example adapted from the official [universat example](https://github.com/gastruc/UniverSat/blob/main/demo.ipynb).

It walks through the full inference surface of UniverSat:
1. Load the pretrained **Base** model via `geoai`.
2. Build a multimodal input (optical + SAR time series + VHR + elevation).
3. Encode at **any output resolution** — patch-level, dense, or per-pixel — from the *same* forward pass.
4. Pool a global tile embedding, run a single modality, and handle an **unseen sensor**.
5. Visualize the dense embeddings with a PCA-to-RGB map.

## 1. Load the model

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../.."))
import torch
import geoai
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = geoai.UniverSatProcessor(device=device)
model = processor.model
# print(model)

In [ ]:
print("params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

## 2. Build a multimodal input

`model.encode(...)` takes a dict `{modality_name -> tensor}` and looks up each sensor's wavelengths, physical resolution, and sub-patch factor from the built-in [`modality_registry.py`](modality_registry.py). Tensor layout:

| Kind | Shape | Extra key |
| --- | --- | --- |
| **Snapshot** (VHR, elevation, …) | `(B, C, H, W)` | — |
| **Time series** (S2, S1, …) | `(B, T, C, H, W)` | `<modality>_dates` of shape `(B, T)` |

Channel count `C` must match the sensor's registered band list (e.g. `spot`=3 RGB, `s2`=10, `s1`=3). Pass **any subset** of modalities. Here we use synthetic tensors so the notebook runs without downloading data — swap in your own rasters to get real embeddings.

In [ ]:
data = {
    "spot": torch.randn(2, 3, 360, 360),  # 1 m VHR, RGB snapshot
    "s2": torch.randn(2, 20, 10, 36, 36),  # 10 m time series
    "s2_dates": torch.randint(0, 365, (2, 20)),
    "s1": torch.randn(2, 12, 3, 36, 36),  # SAR time series
    "s1_dates": torch.randint(0, 365, (2, 12)),
    "dsm": torch.randn(2, 1, 12, 12),  # 30 m elevation snapshot
}
{k: tuple(v.shape) for k, v in data.items()}

data = {k: v.to(device) for k, v in data.items()}

## 3. Encode at any output resolution

Output resolution is **decoupled from the input patch size** — only the requested grid changes. `patch_size` sets the patch size in metres (`patch_size=40` → 40 m patches; `scale = patch_size / 10` internally); `output_grid` is the **side `G`** of the output grid, so a 36×36 feature map is `output_grid=36` (i.e. `G²` tokens).

`encode(...)` returns `(tokens, extras)` where `tokens` has shape `(B, G², embed_dim)` — the spatial grid only. The learnable register tokens the trunk uses internally are stripped off for you.

In [ ]:
def to_grid(tokens):
    """Fold the flat spatial tokens into a (B, G, G, D) feature map.

    `encode(...)` already drops the register tokens, so `tokens` is the pure
    spatial grid of shape (B, G², D)."""
    B, N, D = tokens.shape
    G = int(N**0.5)
    return tokens.reshape(B, G, G, D)


for name, output_grid in [("patch", 9), ("dense", 36), ("high-res", 180)]:
    with torch.no_grad():
        tokens, _ = model.encode(data, patch_size=40, output_grid=output_grid)
    grid = to_grid(tokens)
    print(
        f"{name:6s} output_grid={output_grid:5d}  tokens={tuple(tokens.shape)}  ->  map {tuple(grid.shape)}"
    )

Same model, same inputs — the patch-level transformer runs over a coarse grid, then a *sub-patch skip cross-attention* recovers fine spatial detail at the requested grid (one bilinear resample + one CA pass).

## 4. A global (tile) embedding

For classification / retrieval, mean-pool the spatial grid into a single vector per tile.

In [ ]:
with torch.no_grad():
    tokens, _ = model.encode(data, patch_size=40, output_grid=9)

tile_embed = tokens.mean(dim=1)  # mean-pool the spatial tokens
print("tile embedding:", tuple(tile_embed.shape))

## 5. A single modality

Any subset works — here just the Sentinel-2 time series.

In [ ]:
with torch.no_grad():
    tokens, _ = model.encode(
        {"s2": data["s2"], "s2_dates": data["s2_dates"]},
        patch_size=40,
        output_grid=9,
    )
print("s2-only:", tuple(tokens.shape), "->", tuple(to_grid(tokens).shape))

## 6. An unseen sensor

For a sensor that isn't in the registry, pass its `wavelengths` (optical/hyperspectral) or polarization codes (SAR), `input_res` (m/px), and `subpatches` factor. The UPE uses these as positional encodings — **no retraining**.

In [ ]:
with torch.no_grad():
    tokens, _ = model.encode(
        {
            "mycam": torch.randn(2, 4, 144, 144).to(device)
        },  # 4-band camera, 2.5 m (if aligned to previous example 144 * 2.5 = 360m)
        patch_size=40,
        output_grid=9,
        wavelengths={"mycam": [0.49, 0.56, 0.665, 0.842]},  # B, G, R, NIR (µm)
        input_res={"mycam": 2.5},
        subpatches={"mycam": 1},
    )
print("unseen sensor:", tuple(tokens.shape), "->", tuple(to_grid(tokens).shape))

## 7. Visualize the dense embeddings

Project the per-token feature map to 3 PCA components and render as RGB. With the pretrained checkpoint this surfaces fine spatial structure (field boundaries, roads); with random weights it's just noise — but the pipeline is identical.

In [ ]:
from sklearn.decomposition import PCA

with torch.no_grad():
    tokens, _ = model.encode(data, patch_size=40, output_grid=36)  # 36x36 dense grid
grid = to_grid(tokens)[0]  # (G, G, D)
G, _, D = grid.shape

flat = grid.reshape(G * G, D).float().cpu().numpy()
rgb = PCA(n_components=3).fit_transform(flat).reshape(G, G, 3)
rgb = rgb - rgb.min()
rgb = rgb / (rgb.max() + 1e-8)  # normalize to [0, 1]

plt.figure(figsize=(4, 4))
plt.imshow(rgb)
plt.title(f"PCA(embeddings) — {G}x{G} grid")
plt.axis("off")
plt.show()